### PIIMiddleware

사용자의 입력 문장이나 모델의 출력 문장에서 민감한 개인 식별 정보(PII, Personally Identifiable Information)를 감지하고 안전하게 처리합니다.

- redact: [REDACTED_EMAIL]처럼 개인정보를 완전히 가려버립니다. (가장 안전)
- mask: ****-****-****-4321처럼 마지막 일부만 남기고 가립니다. (고객 응대 시 유용)
- hash: 개인정보를 일관된 해시값(Hash)으로 변환합니다. (데이터 분석을 위해 식별자는 유지하되 원본은 가려야 할 때 유용)
- block: 민감 정보가 감지되는 즉시 에이전트 실행을 중단하고 에러를 발생시킵니다. (API 키 등 절대 유출되면 안 되는 정보에 사용)

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [7]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware

agent = create_agent(
    model="google_genai:gemini-2.5-flash", 
    tools=[],
    middleware=[
        # 이메일은 완전히 가림 (redact)
        PIIMiddleware(pii_type="email", strategy="redact", apply_to_input=True),
        # 신용카드 번호는 마스킹 처리 (mask)
        PIIMiddleware(pii_type="credit_card", strategy="mask", apply_to_input=True),
    ],
)


In [ ]:
prompt = "안녕하세요. 이메일은 kim1@example.com 입니다. 제 카드번호는 1234123443214321 입니다."
response = agent.invoke({"messages": [{"role": "user", "content": prompt}]})

# 에이전트가 실제로 넘겨받은(마스킹된) 메시지 확인
print(response["messages"][0].content)


안녕하세요. 이메일은 [REDACTED_EMAIL] 입니다. 제 카드번호는 ************4321 입니다.


---

In [ ]:
# 휴대폰 번호(010-XXXX-XXXX)를 잡아내는 커스텀 정규식 패턴
phone_number_regex = r"\b(010)[-\s]?(\d{3,4})[-\s]?(\d{4})\b"

phone_masking_middleware = PIIMiddleware(
    pii_type="phone_number",
    detector=phone_number_regex,
    strategy="mask", # 마스킹은 기본적으로 마지막 4자리를 제외하고 마스킹
    apply_to_input=True,
)


In [10]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite", 
    tools=[],
    middleware=[phone_masking_middleware]
)

In [11]:
prompt = "제 전화번호는 010-1234-5678 입니다."

In [12]:
response = agent.invoke({"messages": [{"role": "user", "content": prompt}]})

# 에이전트가 실제로 넘겨받은(마스킹된) 메시지 확인
print(response["messages"][0].content)

제 전화번호는 ****5678 입니다.


---

In [13]:
# 주민등록번호(Resident Registration Number) 정규 표현식 (하이픈 선택 사항)
rrn_detector_regex = r"\b(\d{2}(?:0[1-9]|1[0-2])(?:0[1-9]|[12]\d|3[01]))[-\s]?([1-8]\d{6})\b"

In [15]:
from langchain.agents.middleware import PIIMiddleware

# 커스텀 PII 미들웨어 생성
rrn_masking_middleware = PIIMiddleware(
    pii_type = "rrn_detector_regex", 
    detector=rrn_detector_regex, 
    strategy="mask",
)

In [16]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite", 
    tools=[],
    middleware=[rrn_masking_middleware]
)

In [17]:
prompt = "제 주민등록번호는 990101-1234567 입니다."

response = agent.invoke(
    {"messages": [{"role": "user", "content": prompt}]},
)

In [18]:
print(response["messages"][0].content)

제 주민등록번호는 ****4567 입니다.


### HITL(Human-in-the-loop)

에이전트 도구 호출에 사람의 감독을 추가할 수 있도록 합니다. 

모델이 검토가 필요할 수 있는 작업(예: 파일 쓰기 또는 SQL 실행)을 제안할 경우, 미들웨어는 실행을 일시 중지(인터럽트)하고 결정을 기다릴 수 있습니다.


> https://docs.langchain.com/oss/python/langchain/human-in-the-loop

In [ ]:
from langchain_core.tools import tool

@tool
def write_file_tool(filename: str, content: str) -> str:
    """파일을 지정된 경로에 작성합니다. (이름: write_file)"""
    return f"파일 '{filename}'에 내용이 성공적으로 기록되었습니다."

@tool
def execute_sql_tool(query: str) -> str:
    """데이터베이스에서 SQL 쿼리를 실행합니다. (이름: execute_sql)"""
    return f"쿼리 '{query}'가 실행되었습니다. (영향을 받은 행: 1개)"

@tool
def read_data_tool(source: str) -> str:
    """지정된 소스에서 데이터를 읽어옵니다. (이름: read_data)"""
    return f"'{source}'로부터 데이터를 성공적으로 불러왔습니다: [샘플 데이터]"

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware 
from langgraph.checkpoint.memory import InMemorySaver 


agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    tools=[write_file_tool, execute_sql_tool, read_data_tool],
    middleware=[
        HumanInTheLoopMiddleware( 
            interrupt_on={
                "write_file_tool": True,  # 모든 결정인 approve, edit, reject(승인, 수정, 거절) 허용되는 설정
                "execute_sql_tool": {"allowed_decisions": ["approve", "reject"]},  # 수정을 허용하지 않고 승인 또는 거절만 가능한 설정
                "read_data_tool": False, # 안전한 작업으로 간주하여 사용자 승인 없이 즉시 실행
            },
            description_prefix="Tool execution pending approval",
        ),
    ],
    checkpointer=InMemorySaver(),
    system_prompt="모든 답변은 한국어로 작성해주세요." 
)

In [ ]:
# Human-in-the-loop leverages LangGraph's persistence layer.
# You must provide a thread ID to associate the execution with a conversation thread,
# so the conversation can be paused and resumed (as is needed for human review).
# Run the graph until the interrupt is hit.
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "abc 테이블의 모든 데이터를 삭제해줘",
            }
        ]
    },
    config={"configurable": {"thread_id": "1"}}  
)


In [ ]:
result

{'messages': [HumanMessage(content='abc 테이블의 모든 데이터를 삭제해줘', additional_kwargs={}, response_metadata={}, id='6ea53a66-a3c7-4e80-bba3-9d71cb8a204c'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'execute_sql_tool', 'arguments': '{"query": "DELETE FROM abc"}'}, '__gemini_function_call_thought_signatures__': {'3cf8ea91-6722-431f-83b0-bf886e0770ed': 'CoQDAXLI2nzu1s2C0rroqHtEY1pyWo51HcOEevSx12q2aLFUYKfOheSyiWTVf3M2Bw8UESlojX0Eh1bgQ/nt0nBEyWM6mok2GqncbhI4WYk7Z0mOzsPtqNSQBHAMMtMdRwouS5/kdumXkXluQuhbFTpuj45hMxmy3WDIzgfYZknTb7D4OOBsTS+TuUGcphJDQH7yM0YpX5XBoV+QLxT9IVh3g6J/ymqmLWJleKu7GKCNlmuj1iq1h23A624kSUoxNI9//YDsEDH7X+1plmer44eBsuju7RB358zfl+YCqacCv8RI40A7FaS8OxKC4h9Hn4Vjq5tkDlobRLNbq8q2kw/qw7hdH7AXIhsWvZLKZC+N/qjYW1OsBWfCbEafn17tIrjwtk3w6TTdKjJWmsgpXs/jddVFV5ua3BgtL372UJCjw3gPjp2bALQlwkIVoykJT3HRd/MVBhd5W+JnjM9Z0fTkp7u/yw/M2rU1OnvBztn4KYTCZyUrIB4HlpVSQ4yuG8vFUUBdpQ=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [],

In [ ]:
result['__interrupt__']

[Interrupt(value={'action_requests': [{'name': 'execute_sql_tool', 'args': {'query': 'DELETE FROM abc'}, 'description': "Tool execution pending approval\n\nTool: execute_sql_tool\nArgs: {'query': 'DELETE FROM abc'}"}], 'review_configs': [{'action_name': 'execute_sql_tool', 'allowed_decisions': ['approve', 'reject']}]}, id='f4c4d20ddf70205ca78ccf88763a07ee')]

In [ ]:
from langgraph.types import Command

agent.invoke(
    Command( 
        # resume={"decisions": [{"type": "approve"}]}  # or "reject"
        resume={"decisions": [{"type": "reject"}]}  # or "reject"
    ), 
    config={"configurable": {"thread_id": "1"}} 
)

{'messages': [HumanMessage(content='abc 테이블의 모든 데이터를 삭제해줘', additional_kwargs={}, response_metadata={}, id='6ea53a66-a3c7-4e80-bba3-9d71cb8a204c'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'execute_sql_tool', 'arguments': '{"query": "DELETE FROM abc"}'}, '__gemini_function_call_thought_signatures__': {'3cf8ea91-6722-431f-83b0-bf886e0770ed': 'CoQDAXLI2nzu1s2C0rroqHtEY1pyWo51HcOEevSx12q2aLFUYKfOheSyiWTVf3M2Bw8UESlojX0Eh1bgQ/nt0nBEyWM6mok2GqncbhI4WYk7Z0mOzsPtqNSQBHAMMtMdRwouS5/kdumXkXluQuhbFTpuj45hMxmy3WDIzgfYZknTb7D4OOBsTS+TuUGcphJDQH7yM0YpX5XBoV+QLxT9IVh3g6J/ymqmLWJleKu7GKCNlmuj1iq1h23A624kSUoxNI9//YDsEDH7X+1plmer44eBsuju7RB358zfl+YCqacCv8RI40A7FaS8OxKC4h9Hn4Vjq5tkDlobRLNbq8q2kw/qw7hdH7AXIhsWvZLKZC+N/qjYW1OsBWfCbEafn17tIrjwtk3w6TTdKjJWmsgpXs/jddVFV5ua3BgtL372UJCjw3gPjp2bALQlwkIVoykJT3HRd/MVBhd5W+JnjM9Z0fTkp7u/yw/M2rU1OnvBztn4KYTCZyUrIB4HlpVSQ4yuG8vFUUBdpQ=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [],